In [ ]:
import json, pickle, os
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not set in environment variables"

model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("indexes/faiss_index.bin")

def normalize(scores):
    scores = np.array(scores)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
with open("indexes/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)
with open("indexes/corpus.json") as f:
    corpus = json.load(f)

print(f"Loaded: {len(corpus)} corpus functions")
print(f"FAISS index: {index.ntotal} vectors ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded: 5902 corpus functions
FAISS index: 5902 vectors ✓


In [ ]:
def detect_embedding(query_code, threshold=0.7, k=5):
    """
    Detects plagiarism using cosine similarity between code embeddings.
    Returns plagiarism=True if top similarity score exceeds threshold.
    """
    q_emb = model.encode([query_code], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)

    top_score = float(scores[0][0])
    top_match = corpus[indices[0][0]]

    return {
        "method": "embedding",
        "is_plagiarism": top_score >= threshold,
        "top_similarity": round(top_score, 4),
        "threshold": threshold,
        "top_match_snippet": top_match[:200] + "..."
    }

In [ ]:
def detect_llm(query_code, sample_size=20):
    """
    Provides sampled corpus code directly to LLM context.
    Tests what LLMs can achieve with broad information access.
    """
    import random
    sample = random.sample(corpus, min(sample_size, len(corpus)))
    corpus_text = "\n\n---\n\n".join(sample)

    prompt = f"""You are an expert code plagiarism detector.

Below is a CORPUS of reference code functions, followed by a QUERY function to evaluate.

CORPUS:
{corpus_text}

QUERY CODE:
{query_code}

Determine if the query code is plagiarized from any function in the corpus.
Plagiarism includes: variable renaming, comment removal, minor restructuring, or reformatting while preserving the core logic.

Respond with exactly this format:
VERDICT: YES or NO
REASON: one sentence explanation"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        temperature=0
    )
    answer = response.choices[0].message.content.strip()
    verdict_line = [l for l in answer.splitlines() if l.startswith("VERDICT:")][0]
    is_plagiarism = "YES" in verdict_line.upper()

    return {
        "method": "direct_llm",
        "is_plagiarism": is_plagiarism,
        "raw_response": answer
    }

In [ ]:
def detect_rag(query_code, k=5):

    q_emb = model.encode([query_code], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    scores, indices = index.search(q_emb, k)

    retrieved = [corpus[i] for i in indices[0]]

    
    context = "\n\n---\n\n".join(retrieved[:3])

    prompt = f"""
You are a STRICT code plagiarism detector.

CONTEXT:
{context}

QUERY:
{query_code}

RULES:
- Be conservative
- If unsure → NO

VERDICT: YES or NO
REASON: one sentence
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()
    verdict = "YES" in answer.split("VERDICT:")[1].splitlines()[0].upper()

    return {
        "method": "rag",
        "is_plagiarism": verdict,
        "k": k,
        "raw_response": answer
    }

In [ ]:
def detect_hybrid_rag(query_code, k=5):

   
    q_emb = model.encode([query_code], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    dense_scores, dense_idx = index.search(q_emb, k * 2)

    bm25_raw = np.array(bm25.get_scores(query_code.split()))
    bm25_norm = normalize(bm25_raw)

   
    candidates = []

    for i, idx in enumerate(dense_idx[0]):
        emb = float(dense_scores[0][i])
        bm = float(bm25_norm[idx])

  
        if emb < 0.45 and bm < 0.25:
            continue

        score = 0.65 * emb + 0.35 * bm

        candidates.append({
            "idx": int(idx),
            "emb": emb,
            "bm": bm,
            "score": score
        })

    if not candidates:
        return {
            "method": "hybrid_rag",
            "is_plagiarism": False,
            "reason": "no candidates"
        }

 
    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    
    top = candidates[:3]

   
    evidence = [
        c for c in top
        if (c["emb"] > 0.70 or c["bm"] > 0.30)
    ]

    final = evidence if evidence else top[:1]

    context = "\n\n---\n\n".join([corpus[c["idx"]] for c in final])

    prompt = f"""
You are a STRICT plagiarism verifier.

CONTEXT:
{context}

QUERY:
{query_code}

Rules:
- Be conservative
- If unsure → NO
- Focus on logic similarity, not naming

VERDICT: YES or NO
REASON: one sentence
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()
    verdict = "YES" in answer.split("VERDICT:")[1].splitlines()[0].upper()

    return {
        "method": "hybrid_rag",
        "is_plagiarism": verdict,
        "candidates_used": len(final),
        "raw_response": answer
    }

In [ ]:

test_snippet = '''def bubble_sort_v2(items):
    size = len(items)
    for idx in range(size):
        for jdx in range(size - idx - 1):
            if items[jdx] > items[jdx + 1]:
                items[jdx], items[jdx + 1] = items[jdx + 1], items[jdx]
    return items'''

print("=== Testing all 4 methods ===\n")
for fn in [detect_embedding, detect_llm, detect_rag, detect_hybrid_rag]:
    result = fn(test_snippet)
    verdict = "🚨 PLAGIARISM" if result["is_plagiarism"] else "✅ ORIGINAL"
    print(f"{result['method'].upper()}: {verdict}")
    if "raw_response" in result:
        print(f"  {result['raw_response'][:100]}")
    print()

=== Testing all 4 methods ===

EMBEDDING: 🚨 PLAGIARISM

DIRECT_LLM: ✅ ORIGINAL
  VERDICT: NO  
REASON: The query code implements a bubble sort algorithm, which does not match the fu

RAG: 🚨 PLAGIARISM
  VERDICT: YES  
REASON: The `bubble_sort_v2` function implements the bubble sort algorithm with a sim

HYBRID_RAG: 🚨 PLAGIARISM
  VERDICT: YES  
REASON: The logic of the bubble sort implementation in the query closely resembles th

